# Q2 - Frequent Itemset (Apriori vs FP-Growth vs Compact)
ครอบคลุมข้อ 2.1 และ 2.2 โดยใช้ข้อมูล transactions 1,000 / 3,000 / 5,000 / 10,000

In [1]:
import time
import tracemalloc
import warnings

import pandas as pd
from mlxtend.frequent_patterns import apriori, fpgrowth

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 120)

In [2]:
urls = {
    1000: "https://raw.githubusercontent.com/pakornlee/ml_example/23665225ce5781e8ea782e18829e6108a5a4c92f/transactions_1000_onehot.csv",
    3000: "https://raw.githubusercontent.com/pakornlee/ml_example/23665225ce5781e8ea782e18829e6108a5a4c92f/transactions_3000_onehot.csv",
    5000: "https://raw.githubusercontent.com/pakornlee/ml_example/23665225ce5781e8ea782e18829e6108a5a4c92f/transactions_5000_onehot.csv",
    10000: "https://raw.githubusercontent.com/pakornlee/ml_example/23665225ce5781e8ea782e18829e6108a5a4c92f/transactions_10000_onehot.csv",
}

datasets = {}
for size, url in urls.items():
    df = pd.read_csv(url)
    df.columns = df.columns.str.strip()
    datasets[size] = df.astype(bool)

print("Loaded datasets:", list(datasets.keys()))
print("Shape for N=1000:", datasets[1000].shape)

Loaded datasets: [1000, 3000, 5000, 10000]
Shape for N=1000: (1000, 30)


In [3]:
def measure(func):
    tracemalloc.start()
    start = time.time()
    result = func()
    elapsed = time.time() - start
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return result, elapsed, peak / 1024 / 1024


def run_apriori(df, min_sup, max_len=8):
    return apriori(df, min_support=min_sup, use_colnames=True, max_len=max_len)


def run_fp(df, min_sup, max_len=8):
    return fpgrowth(df, min_support=min_sup, use_colnames=True, max_len=max_len)


def run_compact(df, min_sup, max_k=8, top_n=20):
    total = len(df)
    bit_data = {}

    for col in df.columns:
        bitstring = "".join(df[col].astype(int).astype(str))
        bit_data[frozenset([col])] = int(bitstring, 2)

    levels = {1: {}}
    for itemset, bit in bit_data.items():
        support = bin(bit).count("1") / total
        if support >= min_sup:
            levels[1][itemset] = (bit, support)

    levels[1] = dict(
        sorted(levels[1].items(), key=lambda x: x[1][1], reverse=True)[:top_n]
    )

    for k in range(2, max_k + 1):
        levels[k] = {}
        prev = list(levels[k - 1].keys())
        candidates = set()

        for i in range(len(prev)):
            for j in range(i + 1, len(prev)):
                union = prev[i] | prev[j]
                if len(union) == k:
                    candidates.add(union)

        for c in candidates:
            if all((c - frozenset([x])) in levels[k - 1] for x in c):
                items = list(c)
                bit = bit_data[frozenset([items[0]])]
                for item in items[1:]:
                    bit &= bit_data[frozenset([item])]
                support = bin(bit).count("1") / total
                if support >= min_sup:
                    levels[k][c] = (bit, support)

        if len(levels[k]) == 0:
            break

        levels[k] = dict(
            sorted(levels[k].items(), key=lambda x: x[1][1], reverse=True)[:top_n]
        )

    results = []
    for k, level in levels.items():
        for itemset, (_, sup) in level.items():
            results.append((itemset, sup))

    return results


def count_ge2_apriori(df_result):
    return int((df_result["itemsets"].apply(len) >= 2).sum())


def count_ge2_compact(compact_result):
    return sum(1 for itemset, _ in compact_result if len(itemset) >= 2)

In [4]:
# 2.1 Algorithm comparison (N=1000, min_sup=0.05)
min_sup = 0.05
df = datasets[1000]

ap_result, ap_time, ap_mem = measure(lambda: run_apriori(df, min_sup, max_len=6))
fp_result, fp_time, fp_mem = measure(lambda: run_fp(df, min_sup, max_len=6))
cp_result, cp_time, cp_mem = measure(lambda: run_compact(df, min_sup, max_k=6))

table_21 = pd.DataFrame(
    [
        ["Apriori", len(ap_result), ap_time, ap_mem],
        ["FP-Growth", len(fp_result), fp_time, fp_mem],
        ["Compact", len(cp_result), cp_time, cp_mem],
    ],
    columns=["algorithm", "frequent_itemsets_count", "time_sec", "peak_memory_mb"],
)
print("Q2.1 - Algorithm Comparison")
print(table_21.to_string(index=False))

Q2.1 - Algorithm Comparison
algorithm  frequent_itemsets_count  time_sec  peak_memory_mb
  Apriori                       90  0.020481        0.565029
FP-Growth                       90  0.075846        0.225555
  Compact                       44  0.200068        0.081612


In [5]:
# 2.2.1 Scaling by number of transactions (min_sup=0.05)
rows = []
for size in [1000, 3000, 5000, 10000]:
    df = datasets[size]

    ap_res, ap_t, ap_m = measure(lambda: run_apriori(df, 0.05, max_len=6))
    fp_res, fp_t, fp_m = measure(lambda: run_fp(df, 0.05, max_len=6))
    cp_res, cp_t, cp_m = measure(lambda: run_compact(df, 0.05, max_k=6))

    rows.extend(
        [
            [size, "Apriori", len(ap_res), ap_t, ap_m],
            [size, "FP-Growth", len(fp_res), fp_t, fp_m],
            [size, "Compact", len(cp_res), cp_t, cp_m],
        ]
    )

table_221 = pd.DataFrame(
    rows,
    columns=["transactions", "algorithm", "frequent_itemsets_count", "time_sec", "peak_memory_mb"],
)
print("Q2.2.1 - Scaling by transaction count")
print(table_221.to_string(index=False))

Q2.2.1 - Scaling by transaction count
 transactions algorithm  frequent_itemsets_count  time_sec  peak_memory_mb
         1000   Apriori                       90  0.017979        0.555714
         1000 FP-Growth                       90  0.062433        0.222610
         1000   Compact                       44  0.169837        0.077392
         3000   Apriori                       89  0.017441        1.681328
         3000 FP-Growth                       89  0.150614        0.310040
         3000   Compact                       42  0.505402        0.212203
         5000   Apriori                       90  0.019961        2.769882
         5000 FP-Growth                       90  0.235007        0.347313
         5000   Compact                       41  0.815986        0.342837
        10000   Apriori                       89  0.023592        5.541910
        10000 FP-Growth                       89  0.452377        0.637966
        10000   Compact                       41  1.610461    

In [6]:
# 2.2.2 Vary min_support at N=1000 (count only itemsets with size >= 2)
rows = []
df = datasets[1000]

for ms in [0.01, 0.05, 0.1, 0.2]:
    ap_res, ap_t, ap_m = measure(lambda: run_apriori(df, ms, max_len=6))
    fp_res, fp_t, fp_m = measure(lambda: run_fp(df, ms, max_len=6))
    cp_res, cp_t, cp_m = measure(lambda: run_compact(df, ms, max_k=6))

    rows.extend(
        [
            [ms, "Apriori", count_ge2_apriori(ap_res), ap_t, ap_m],
            [ms, "FP-Growth", count_ge2_apriori(fp_res), fp_t, fp_m],
            [ms, "Compact", count_ge2_compact(cp_res), cp_t, cp_m],
        ]
    )

table_222 = pd.DataFrame(
    rows,
    columns=["min_support", "algorithm", "count_ge_2_itemsets", "time_sec", "peak_memory_mb"],
)
print("Q2.2.2 - Min support variation (>=2-itemset)")
print(table_222.to_string(index=False))

Q2.2.2 - Min support variation (>=2-itemset)
 min_support algorithm  count_ge_2_itemsets  time_sec  peak_memory_mb
        0.01   Apriori                  417  0.044719        1.626948
        0.01 FP-Growth                  417  0.094179        0.528029
        0.01   Compact                   36  0.174940        0.077964
        0.05   Apriori                   79  0.017051        0.554909
        0.05 FP-Growth                   79  0.060850        0.256775
        0.05   Compact                   33  0.172287        0.077598
        0.10   Apriori                   24  0.011281        0.358757
        0.10 FP-Growth                   24  0.056212        0.165239
        0.10   Compact                   22  0.170259        0.078027
        0.20   Apriori                    6  0.007706        0.155235
        0.20 FP-Growth                    6  0.049684        0.121517
        0.20   Compact                    6  0.174772        0.077455


In [7]:
# 2.2.3 Vary max_itemset k at N=1000, min_sup=0.05 (count only >=2-itemset)
rows = []
df = datasets[1000]

for k in [4, 5, 6, 7, 8]:
    ap_res, ap_t, ap_m = measure(lambda: run_apriori(df, 0.05, max_len=k))
    fp_res, fp_t, fp_m = measure(lambda: run_fp(df, 0.05, max_len=k))
    cp_res, cp_t, cp_m = measure(lambda: run_compact(df, 0.05, max_k=k))

    rows.extend(
        [
            [k, "Apriori", count_ge2_apriori(ap_res), ap_t, ap_m],
            [k, "FP-Growth", count_ge2_apriori(fp_res), fp_t, fp_m],
            [k, "Compact", count_ge2_compact(cp_res), cp_t, cp_m],
        ]
    )

table_223 = pd.DataFrame(
    rows,
    columns=["max_k", "algorithm", "count_ge_2_itemsets", "time_sec", "peak_memory_mb"],
)
print("Q2.2.3 - Max itemset (k) variation (>=2-itemset)")
print(table_223.to_string(index=False))

Q2.2.3 - Max itemset (k) variation (>=2-itemset)
 max_k algorithm  count_ge_2_itemsets  time_sec  peak_memory_mb
     4   Apriori                   79  0.016780        0.555683
     4 FP-Growth                   79  0.064300        0.220901
     4   Compact                   33  0.171899        0.078298
     5   Apriori                   79  0.015908        0.554909
     5 FP-Growth                   79  0.059851        0.257484
     5   Compact                   33  0.179152        0.081737
     6   Apriori                   79  0.016997        0.554909
     6 FP-Growth                   79  0.060183        0.256844
     6   Compact                   33  0.170120        0.077359
     7   Apriori                   79  0.015930        0.554909
     7 FP-Growth                   79  0.060181        0.256844
     7   Compact                   33  0.172109        0.077788
     8   Apriori                   79  0.015371        0.554909
     8 FP-Growth                   79  0.061049        